# Step-by-step implementation
The following are the steps to implement the decomposition technique:
1. Import necessary libraries
2. Set up the LangSmith and OpenAI API keys
3. Load document
4. Split text
5. Index documents
6. Create sub-questions: Decompose the main question
7. Generate answers for sub-questions
8. Merge sub-questions and answers
9. Generate the final answer

## 1. Import necessary libraries

In [25]:
import os
import bs4
from pydantic import BaseModel, Field
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from helpers import get_experientiallabs_llm
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langsmith import Client
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

## 2. Set up the LangSmith and OpenAI API keys

In [26]:
from dotenv import load_dotenv
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [27]:
# LANGSMITH_PROJECT (not the legacy LANGCHAIN_PROJECT): the SDK checks the
# LANGSMITH_ prefix FIRST, so a LANGSMITH_PROJECT in .env would silently win
# and these traces would land in that project instead of this one.
os.environ['LANGSMITH_PROJECT'] = 'Decomposition'

## 3. Load document

In [28]:
# provide you document
loader = WebBaseLoader(
    web_paths=("https://blog.langchain.dev/announcing-langsmith",),
)

blog_docs = loader.load()

## 4. Split text

In [29]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=100,
    chunk_overlap=20)

splits = text_splitter.split_documents(blog_docs)

## 5. Index documents

In [30]:
vectorstore = Chroma.from_documents(documents=splits, embedding=OpenAIEmbeddings())

retriever = vectorstore.as_retriever()

## 6. Create sub-questions: Decompose the main question

### Why structured output instead of `split("\n")`?

An earlier version of this cell asked for prose and parsed it by splitting on newlines:

```python
generate_queries_decomposition = (
    prompt_decomposition | llm | StrOutputParser() | (lambda x: x.split("\n"))
)
```

That fails in two compounding ways.

**1. The prompt asked for two different things.** It said *"break the input into three
sub-questions"* and then *"generate multiple search queries"*. The model obeyed both and
returned nested markdown — sub-questions **plus** a list of search queries, under headers.
That is compliance, not misbehavior. The original tutorial got away with it because weaker
models ignored the second instruction.

**2. `split("\n")` trusts the model's formatting.** Splitting that markdown yields ~18 list
items: headers like `'### Sub-questions'`, blank strings, and bullet lines. The loop in the
next section then calls `retriever.invoke()` **and** an LLM on every one of them — roughly
18 retrievals and 18 LLM calls instead of 3, most on meaningless input like `''`. Those
garbage answers then flow into the final synthesis step.

Worse, it is *non-deterministic*: the same chain sometimes returns 3 clean lines and
sometimes 18, so the bug appears and disappears between runs.

### The fix

`with_structured_output()` binds a Pydantic schema to the model, so the provider returns
validated JSON that LangChain parses into a `SubQuestions` object. You get a guaranteed
`list[str]` — no headers, no blanks, no prompt-wording roulette.

| | String parsing | Structured output |
|---|---|---|
| **Output type** | Whatever the model emitted | Guaranteed `list[str]` |
| **Formatting drift** | Breaks silently | Impossible — schema-validated |
| **Prompt burden** | Must specify exact layout | Only needs to describe the *task* |
| **Failure mode** | Wrong results, extra cost | Raises immediately |

> **Rule of thumb**: any time you are about to `.split()` or regex an LLM response to get a
> list or a record, reach for `with_structured_output()` instead.

In [31]:
class SubQuestions(BaseModel):
    """Three independent sub-questions that together cover the original question."""

    questions: list[str] = Field(
        description="Exactly three sub-questions, each answerable on its own"
    )


# The prompt now asks for ONE thing. Formatting is the schema's job, so there is
# no need to beg the model for "one per line, no markdown, no headers".
template = """Break the input question into exactly three sub-questions.

Each sub-question must be answerable independently, without needing the answers to
the others, and together they should fully cover the original question.

Original question: {question}"""

prompt_decomposition = ChatPromptTemplate.from_template(template)

llm = get_experientiallabs_llm()

generate_queries_decomposition = (
    prompt_decomposition
    | llm.with_structured_output(SubQuestions)
    | (lambda x: x.questions)
)

question = "What is LangSmith, and why do we need it?"

result = generate_queries_decomposition.invoke({"question": question})

In [32]:
result

['What is LangSmith, and what are its main features and purpose?',
 'What problems in developing, debugging, testing, and monitoring LLM applications does LangSmith address?',
 'Why would a team need LangSmith, and what benefits does it provide compared with building or managing these capabilities independently?']

## 7. Generate answers for sub-questions

In [34]:
prompt_rag = Client().pull_prompt("rlm/rag-prompt", dangerously_pull_public_prompt=True)

sub_questions = generate_queries_decomposition.invoke({"question":question})
rag_results = []
for sub_question in sub_questions:
  retrieved_docs = retriever.invoke(sub_question)
  answer = (prompt_rag | llm | StrOutputParser()).invoke({"context": retrieved_docs,
                                                                "question": sub_question})
  rag_results.append(answer)

In [35]:
rag_results

['LangSmith is a unified platform for debugging, testing, evaluating, and monitoring large language model (LLM) applications, helping teams move from prototypes to production. Its capabilities include tracing model inputs and outputs at every chain step, experimenting with prompts and chains, and identifying unexpected results, errors, or latency.',
 'LangSmith helps teams debug LLM applications by providing visibility into every step’s inputs and outputs, making it easier to identify unexpected results, errors, and latency. It unifies experimentation, testing, evaluation, and production monitoring so teams can validate changes, track performance, and troubleshoot issues as applications move from prototype to production.',
 'LangSmith provides a unified platform for debugging, testing, evaluating, and monitoring LLM applications, with visibility into each chain step to identify errors, unexpected outputs, and latency. It also supports prompt and chain experimentation, automated evaluat

## 8. Merge sub-questions and answers

In [0]:
def format(questions, answers):
    """Format Q and A"""

    formatted_string = ""
    for i, (question, answer) in enumerate(zip(questions, answers), start=1):
        formatted_string += f"Question {i}: {question}\nAnswer {i}: {answer}\n\n"
    return formatted_string.strip()

context = format(sub_questions, rag_results)

## 9. Generate the final answer

In [0]:
template = """Here is a set of Q and A:

{context}

Use these to synthesize an answer to the question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

final_rag_chain = (
    prompt
    | llm
    | StrOutputParser()
)

final_rag_chain.invoke({"context":context,"question":question})